In [44]:
import pandas as pd
import numpy as np
import googlemaps
from ares import logger
import os
from pathlib import Path
from dotenv import load_dotenv
from ares.utils.common import load_json, save_json

df = pd.read_csv("../artifacts/data/01-raw/untouched_raw_original.csv")
train_df = pd.read_csv("../artifacts/data/01-raw/train_df.csv")
eval_df = pd.read_csv("../artifacts/data/01-raw/eval_df.csv")
geocode_path = Path("../artifacts/data/04-geocode_cache/geocode_cache.json")

load_dotenv()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [45]:
print(train_df.shape)
print(eval_df.shape)

(12017, 30)
(3005, 30)


# Geocode localities


In [53]:
df.locality.value_counts().sort_index()

locality
Abelemkpe                     69
Ablekuma                     206
Abokobi                       87
Accra Metropolitan           847
Achimota                     415
Adabraka                       2
Adenta                       849
Adjiriganor                  538
Agbogba                       66
Airport Residential Area     566
Alajo                          8
Amrahia                       30
Anyaa                         36
Ashaiman Municipal           120
Ashaley Botwe                583
Ashomang Estate              127
Asylum Down                    6
Awoshie                       48
Burma Camp                    43
Cantonments                  389
Circle                         3
Danfa                         34
Dansoman                     199
Darkuman                       8
Dodowa                        33
Dome                         357
Dzorwulu                     208
East Legon                  1707
Ga East Municipal            157
Ga South Municipal           104
G

In [47]:
maps = googlemaps.Client(key=os.getenv("GOOGLE_MAPS_KEY"))

[2025-12-31 18:11:04,968: INFO: client: API queries_quota: 60]


In [48]:
def geocode(x):
    """Geocode a location x using Google Maps API"""
    try:
        result = maps.geocode(x.lower(), region="gh")
        if result:
            lat = result[0]["geometry"]["location"]["lat"]
            lng = result[0]["geometry"]["location"]["lng"]
            return (lat, lng)
    except Exception as e:
        logger.info(f"Error geocoding {x}: {e}")
    return (None, None)


def create_cache(df):
    cache = {}
    locs = df["locality"].unique()
    for i, loc in enumerate(locs, 1):
        lat, lng = geocode(loc)
        cache[loc] = {"lat": lat, "lng": lng}

    save_json(geocode_path, cache)
    logger.info(f"{len(cache)} geocodes cached successfuly.")
    return cache

In [49]:
def get_lat_lng(location):
    """Fetch the latitude and longitude of a location"""
    try:
        geocodes_cache = load_json(geocode_path)
        logger.info(f"{len(geocodes_cache)} geocodes loaded successfully")
    except FileNotFoundError:
        logger.info("Cache not found, building cache")
        geocode_path.parent.mkdir(parents=True, exist_ok=True)
        geocodes_cache = create_cache(df)

    location_lower = location.lower()
    if location_lower in geocodes_cache:
        result = geocodes_cache[location_lower]
        return (result["lat"], result["lng"])
    else:
        logger.warning(f"Location '{location}' not in cache, geocoding")
        lat, lng = geocode(location_lower)
        geocodes_cache[location_lower] = {"lat": lat, "lng": lng}
        save_json(geocode_path, geocodes_cache)
        return (lat, lng)


In [50]:
get_lat_lng("Greater Accra, Amasaman, Abease")

[2025-12-31 18:11:05,085: INFO: common: JSON file loaded successfully from: ../artifacts/data/04-geocode_cache/geocode_cache.json]
[2025-12-31 18:11:05,092: INFO: 586160306: 74 geocodes loaded successfully]


(5.7038601, -0.3085652)